# Milestone -2 
## Classical ML Model

In [1]:
import os
import random
import numpy as np
import pandas as pd

import librosa
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

In [2]:
BASE = "/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup"

GENRES_PATH = os.path.join(BASE, "genres_stems")
MASHUPS_PATH = os.path.join(BASE, "mashups")
TEST_CSV = os.path.join(BASE, "test.csv")

print("BASE:", BASE)
print("BASE folders:", os.listdir(BASE))

BASE: /kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup
BASE folders: ['ESC-50-master', 'sample_submission.csv', 'genres_stems', 'mashups', 'test.csv']


In [3]:
genres = sorted(os.listdir(GENRES_PATH))

label_map = {g:i for i,g in enumerate(genres)}
inv_map = {i:g for g,i in label_map.items()}

print(genres)

['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']


In [4]:
def load_song(song_folder):

    stems = ["drums.wav","bass.wav","vocals.wav","other.wav"]

    audio_sum = None
    sr = 22050

    for stem in stems:

        path = os.path.join(song_folder, stem)

        if not os.path.exists(path):
            continue

        try:
            y, sr = librosa.load(path, sr=sr)
        except:
            continue

        if audio_sum is None:
            audio_sum = y
        else:
            min_len = min(len(audio_sum), len(y))
            audio_sum = audio_sum[:min_len] + y[:min_len] * 0.25

    if audio_sum is None:
        audio_sum = np.zeros(sr*5)

    return audio_sum, sr

In [5]:
def create_mashup(genre_folder):

    stems = ["drums.wav","bass.wav","vocals.wav","other.wav"]

    songs = os.listdir(genre_folder)

    selected = random.sample(songs,4)

    audio_sum = None
    sr = 22050

    for song,stem in zip(selected,stems):

        path = os.path.join(genre_folder,song,stem)

        if not os.path.exists(path):
            continue

        try:
            y,sr = librosa.load(path,sr=sr)
        except:
            continue

        if audio_sum is None:
            audio_sum = y
        else:
            min_len = min(len(audio_sum),len(y))
            audio_sum = audio_sum[:min_len] + y[:min_len]*0.25

    if audio_sum is None:
        audio_sum = np.zeros(sr*5)

    return audio_sum,sr

In [6]:
def extract_features(audio, sr):

    audio = librosa.util.normalize(audio)

    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=sr,
        n_mfcc=60
    )

    mfcc_mean = np.mean(mfcc, axis=1)
    mfcc_std = np.std(mfcc, axis=1)

    features = np.concatenate([mfcc_mean, mfcc_std])

    return features

In [7]:
X = []
y = []

for genre in genres:

    genre_path = os.path.join(GENRES_PATH,genre)

    for _ in tqdm(range(60)):   # mashups per genre

        audio,sr = create_mashup(genre_path)

        feat = extract_features(audio,sr)

        X.append(feat)
        y.append(label_map[genre])

X = np.array(X)
y = np.array(y)

print(X.shape)

100%|██████████| 60/60 [00:31<00:00,  1.89it/s]

(600, 120)


In [8]:
X_train,X_val,y_train,y_val = train_test_split(
    X,y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [9]:
model = LogisticRegression(
    max_iter=3000,
    solver="saga",
    class_weight="balanced",
    n_jobs=-1
)

In [10]:
model.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


LogisticRegression(class_weight='balanced', max_iter=3000, n_jobs=-1,
                   solver='saga')

In [11]:
pred = model.predict(X_val)

score = f1_score(y_val,pred,average="macro")

print("Validation Macro F1:",score)

Validation Macro F1: 0.7135189995189994


In [12]:
test_df = pd.read_csv(TEST_CSV)

test_features = []

for file in tqdm(test_df["id"]):

    path = os.path.join(MASHUPS_PATH,f"song{file:04d}.wav")

    audio,sr = librosa.load(path,sr=22050)

    feat = extract_features(audio,sr)

    test_features.append(feat)

test_features = np.array(test_features)

100%|██████████| 3020/3020 [04:19<00:00, 11.64it/s]


In [13]:
preds = model.predict(test_features)

genres_pred = [inv_map[p] for p in preds]

In [14]:
submission = pd.DataFrame({
    "id": test_df["id"],
    "genre": genres_pred
})

submission.to_csv("submission.csv",index=False)

submission.head()

,id,genre
0,1,pop
1,2,classical
2,3,disco
3,4,rock
4,5,rock
